<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Batches, epochs and validation

**[Batches, epochs and validation](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/)** · Deep Learning and Transformers Explained · Module 3, lesson 5 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** train in shuffled mini-batches over several epochs, measure on the validation tickets in evaluation mode, read the training and validation curves, and keep the best epoch.

| | |
|---|---|
| **Time** | About 35 minutes, with the module practice |
| **Needs** | An internet connection for the setup cells. No account, no graphics card. Each training run takes a few seconds on a CPU. |
| **You should know** | The five steps of [The training loop](https://learning.nextia-ai.com/courses/deep-learning/m03/the-training-loop/). Training, validation and test sets, from [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/) in the machine learning course. |
| **Tested** | 2026-10-08, Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M03-L05-batches-epochs-and-validation/batches-epochs-and-validation-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: the tickets as numbers

The next cell turns the tickets into word counts, as in [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/): a vocabulary from the training tickets, then one row of counts per ticket. It also makes `loss_fn`, the loss of this module. Run it.

> **Check.** You should see `x_train shape (4696, 2864)`, `x_valid shape (600, 2864)` and `vocabulary: 2864`.

In [ ]:
import torch
from torch import nn

# The tickets as numbers: a vocabulary from the training tickets, then word counts.
vocab = Vocab([r["text"] for r in train_rows])
x_train, y_train = bag_of_words(train_rows, vocab), labels(train_rows)
x_valid, y_valid = bag_of_words(valid_rows, vocab), labels(valid_rows)
loss_fn = nn.CrossEntropyLoss()
show_shape("x_train", x_train)
show_shape("x_valid", x_valid)
print("vocabulary:", len(vocab), "| teams:", TEAMS)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Batches

A **mini-batch** (or **batch**) is a small group of training tickets, here 64. Each training step uses one batch instead of all 4,696 tickets. One pass through all the batches, so through every training ticket once, is one **epoch**.

> **Predict.** How many batches of 64 make one epoch of 4,696 tickets, and how many tickets are in the last batch? Then run the cell.

In [ ]:
batch_size = 64
order = torch.randperm(len(x_train))                 # the row numbers 0 ... 4695, shuffled
starts = list(range(0, len(order), batch_size))
print("batches per epoch:", len(starts))
show_shape("batch 1", x_train[order[0:64]])
show_shape("last batch", x_train[order[starts[-1]:starts[-1] + batch_size]])

> **Check.** You should see `batches per epoch: 74`, `batch 1 shape (64, 2864)` and `last batch shape (24, 2864)`.

`torch.randperm(n)` gives the numbers 0 to n − 1 in a random order. `x_train[order[0:64]]` picks those 64 rows. 73 full batches hold 4,672 tickets, so the last batch has the other 24.

## 2. The loop with batches and validation

The training loop of the last lesson gets two changes: an inner loop over the batches, and a measurement on the validation tickets after each epoch. The measurement uses `model.eval()` and `torch.no_grad()`:

- `model.train()` and `model.eval()` switch the **mode** of the network. Some layers act differently while training. The ticket network has a **dropout** layer (switched off with `dropout=0.0` here), which drops random neurons during training only. `eval()` switches such behaviour off.
- `with torch.no_grad():` tells PyTorch to keep no record for gradients inside the block: less memory, more speed, and no way to train by accident.

> **Predict.** The full-batch loop of the last lesson needed 100 steps (100 passes over the data) for 97.8%. What will one epoch of batches give? Then run the cell.

In [ ]:
set_seed(0)
model = BagOfWordsNet(len(vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
batch_size = 64

for epoch in range(1, 4):
    model.train()                                    # training mode
    order = torch.randperm(len(x_train))             # shuffle: a new order every epoch
    for start in range(0, len(order), batch_size):
        batch = order[start:start + batch_size]      # the row numbers of one batch
        optimizer.zero_grad()
        loss = loss_fn(model(x_train[batch]), y_train[batch])
        loss.backward()
        optimizer.step()
    model.eval()                                     # evaluation mode: dropout off
    with torch.no_grad():                            # no record, no gradients
        scores = model(x_valid)
        valid_loss = loss_fn(scores, y_valid).item()
        valid_acc = (scores.argmax(dim=1) == y_valid).float().mean().item()
    print(f"epoch {epoch}   validation loss {valid_loss:.4f}   accuracy {valid_acc:.4f}")

> **Check.** You should see three lines: epoch 1 with validation loss `0.5486` and accuracy `0.9717`, epoch 2 with `0.2065` and `0.9867`, epoch 3 with `0.1374` and `0.9850`.

One epoch of 74 small steps reaches 97.2%. Full-batch training with one step per pass got 33.7% after one pass (in the lesson's runs): the batches give 74 updates for the same reading of the data.

## 3. Why shuffle?

Suppose the training file were sorted by team: all delivery tickets first, then returns, and so on. Without shuffling, every batch would hold one team only.

> **Predict.** After one epoch on the sorted tickets without shuffling, will the validation accuracy be near 97%, or much lower? Then run the cell.

In [ ]:
by_team = torch.argsort(y_train, stable=True)        # the rows sorted by team: delivery first, account last
set_seed(0)
model = BagOfWordsNet(len(vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.train()
for start in range(0, len(by_team), 64):            # no shuffle: the batches come in team order
    batch = by_team[start:start + 64]
    optimizer.zero_grad()
    loss_fn(model(x_train[batch]), y_train[batch]).backward()
    optimizer.step()
print("sorted, not shuffled:", evaluate(model, x_valid, y_valid))

> **Check.** You should see a validation accuracy of about `0.405`: 40.5%, against 97% with shuffling.

Each run of one-team batches pulls the network towards that team, and the next team undoes it. **Shuffle the training rows in every epoch.** The order of the validation rows does not matter, because evaluation changes nothing.

## 4. The full run: training and validation curves

The course's `train()` function in `ticketnet.py` is the loop above, with `evaluate()` on the training and the validation tickets after each epoch. It returns one row per epoch.

> **Predict.** Over 20 epochs, will the validation loss fall at every epoch, like the training loss? Then run the cell. It takes about 10 seconds.

In [ ]:
set_seed(0)
model = BagOfWordsNet(len(vocab))
history = train(model, x_train, y_train, x_valid, y_valid, epochs=20, log=None)
for row in history:
    print(row)

> **Check.** You should see 20 rows. The training loss falls at every epoch, from `0.5345` to `0.0059`. The validation loss falls to `0.0633` at epoch 12, then rises to `0.0723` at epoch 20. The training accuracy reaches `0.9996`.

Run the next cell to draw both loss curves.

In [ ]:
import matplotlib.pyplot as plt

epochs = [row["epoch"] for row in history]
plt.figure(figsize=(7, 3))
plt.plot(epochs, [row["train_loss"] for row in history], marker="o", label="training loss")
plt.plot(epochs, [row["valid_loss"] for row in history], marker="o", label="validation loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

> **Check.** You should see the training loss go down to almost 0. The validation loss goes down with it at first, then stays flat and rises a little after epoch 12.

The gap after epoch 12 is **overfitting**: the network learns the training tickets by heart, including the 70 tickets with a wrong team, and gets a little worse on new tickets. The **best epoch** is the one with the lowest validation loss.

> **Your turn.** Find the best epoch: the row of `history` with the lowest `"valid_loss"`. Put the row in `best`. Then run the check cell.

In [ ]:
best = min(history, key=lambda row: row["valid_loss"])
print(best)

In [ ]:
expect('the best epoch', best['epoch'] if isinstance(best, dict) else best, 12)

## 5. Failure case: measuring with dropout on

Tomás measures a network that has a dropout layer, but forgets `model.eval()`. The next cell trains a network with dropout 0.5 for 8 epochs, then asks it about the same validation ticket three times in training mode and three times in evaluation mode. It prints the probability of the right team, payment.

> **Predict.** Will the three answers in training mode be the same? Then run the cell.

In [ ]:
set_seed(0)
model = BagOfWordsNet(len(vocab), dropout=0.5)
train(model, x_train, y_train, x_valid, y_valid, epochs=8, log=None)
one_ticket = x_valid[:1]
for mode in ["train", "eval"]:
    model.train() if mode == "train" else model.eval()
    with torch.no_grad():
        answers = [round(torch.softmax(model(one_ticket), dim=1)[0, 2].item(), 3) for _ in range(3)]
    print(f"{mode:5} mode, probability of payment, three times: {answers}")

> **Check.** In training mode you should see three different numbers, for example `[0.996, 1.0, 0.983]`. In evaluation mode you should see the same number three times, `[0.999, 0.999, 0.999]`.

In training mode, dropout removes a different random half of the hidden neurons at every pass, so the same ticket gets a different answer each time. **Answers that change between runs are the symptom.** **Fix:** call `model.eval()` before you measure or use a network, and `model.train()` before you train it again.

## 6. Module practice: a smaller network, logged

> **Your turn.** Train `BagOfWordsNet(len(vocab), hidden=32)` from `set_seed(0)` for 10 epochs with `train()`. Keep the returned rows in `practice`. Then find the best epoch by validation loss and put that row in `practice_best`. Run the check cell.

In [ ]:
set_seed(0)
small = BagOfWordsNet(len(vocab), hidden=32)
print("parameters:", count_parameters(small))
practice = train(small, x_train, y_train, x_valid, y_valid, epochs=10, log=None)
for row in practice:
    print(row)
practice_best = min(practice, key=lambda row: row["valid_loss"])
print("best:", practice_best)

In [ ]:
expect_hash('practice_best', practice_best, '2a991253b82cd919')

Compare it with the 64-neuron network: how many parameters, which best epoch, and which validation loss? The lesson page has a model answer.

## Next

You can now train in shuffled mini-batches, measure on the validation tickets in evaluation mode, and read the two curves to choose the best epoch. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/) has the module check, including the completion check: tell a training step from an evaluation pass. Module 4 diagnoses training problems: learning rates, overfitting and bugs.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Learning rate and optimizers](https://learning.nextia-ai.com/courses/deep-learning/m04/learning-rate-and-optimizers/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them.